# 🗂️ Notebook 2: Notification System — Data Model, API & Mini Pipeline

Here we turn the architecture into something runnable. By the end you'll have:

- a SQLite schema for templates, preferences, and a send log,
- a Pydantic-validated `POST /notify` request model,
- a **tiny end-to-end pipeline** that renders a template, checks preferences, writes a send-log row, and 'delivers' to a fake provider — all in ~60 lines of code.

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

| Table | Columns | Purpose |
|---|---|---|
| `templates` | `id, name, body_template, channel` | Reusable text like `"Hi {name}, your order {order_id} shipped!"` |
| `prefs` | `user_id, channel, enabled, quiet_start, quiet_end` | Opt-outs and quiet hours per user & channel |
| `send_log` | `id, user_id, channel, dedup_key, status, ts` | Durable record of every send attempt (also used for dedup) |
| `device_tokens` | `user_id, platform, token` | Where to push on mobile |

### Why a send_log?

1. **Idempotency** — we check `dedup_key` before sending again.
2. **Audit** — customer support can answer "did we send the receipt?".
3. **Analytics** — delivery rate, retries, cost per channel.

In [ ]:
# A runnable SQLite schema you can poke at.
import sqlite3, os

DB = "/tmp/notif_lab.db"
if os.path.exists(DB): os.remove(DB)

con = sqlite3.connect(DB)
con.executescript('''
CREATE TABLE templates (
  id INTEGER PRIMARY KEY,
  name TEXT UNIQUE,
  channel TEXT,
  body_template TEXT
);
CREATE TABLE prefs (
  user_id INTEGER,
  channel TEXT,
  enabled INTEGER DEFAULT 1,
  quiet_start INTEGER,           -- hour 0-23, NULL = no quiet hours
  quiet_end   INTEGER,
  PRIMARY KEY (user_id, channel)
);
CREATE TABLE send_log (
  id INTEGER PRIMARY KEY AUTOINCREMENT,
  user_id INTEGER,
  channel TEXT,
  dedup_key TEXT UNIQUE,         -- UNIQUE gives us idempotency for free
  status TEXT,
  ts REAL
);
''')
# seed a template + a user who disabled marketing push
con.execute("INSERT INTO templates(name,channel,body_template) VALUES (?,?,?)",
            ("order_shipped", "push", "Hi {name}, your order {order_id} shipped!"))
con.execute("INSERT INTO prefs(user_id,channel,enabled) VALUES (?,?,?)", (42,"push",1))
con.execute("INSERT INTO prefs(user_id,channel,enabled) VALUES (?,?,?)", (99,"push",0))
con.commit()
print("seeded:", con.execute("SELECT name FROM templates").fetchall())


## The API

Callers send a single JSON body. We validate it with **Pydantic** (catches bad inputs
before they touch the DB or queue).

```http
POST /notify
{
  "user_id": 42,
  "template": "order_shipped",
  "channel": "push",
  "priority": "normal",
  "dedup_key": "order-123-shipped",
  "vars": { "name": "Alice", "order_id": 123 }
}
```

**Response: `202 Accepted`** with a `send_id`. We've durably queued it — actual delivery
happens asynchronously.

In [ ]:
from pydantic import BaseModel, Field
from typing import Literal

class NotifyRequest(BaseModel):
    user_id: int
    template: str
    channel: Literal["push","email","sms"]
    priority: Literal["high","normal","low"] = "normal"
    dedup_key: str = Field(min_length=1, max_length=200)
    vars: dict = {}

ok = NotifyRequest(user_id=42, template="order_shipped",
                   channel="push", dedup_key="order-123-shipped",
                   vars={"name":"Alice","order_id":123})
print(ok.model_dump_json(indent=2))

# Pydantic rejects invalid requests for us:
from pydantic import ValidationError
try:
    NotifyRequest(user_id=42, template="x", channel="carrier-pigeon",
                  dedup_key="k", vars={})
except ValidationError as e:
    print("\nRejected bad channel:")
    print(e.errors()[0]["msg"])


## End-to-end mini pipeline

Below is a ~60-line toy version of the whole thing:

```
request → validate → check prefs → render template → dedup → log → deliver
```

In production each arrow might be a network hop (API → Kafka → worker → provider),
but the logic is identical. Running it makes the flow concrete.

In [ ]:
import time

def render(body: str, vars: dict) -> str:
    # super-simple mustache-less templating; production would use Jinja2.
    return body.format(**vars)

def fake_provider_send(channel: str, user_id: int, body: str) -> str:
    # Pretend we called APNs / Twilio / SMTP here.
    print(f"  [{channel}] → user {user_id}: {body}")
    return "delivered"

def notify(req: NotifyRequest) -> dict:
    # 1. load template
    row = con.execute("SELECT body_template, channel FROM templates WHERE name=?",
                      (req.template,)).fetchone()
    if not row:
        return {"status": "error", "reason": "unknown_template"}
    body_tmpl, tmpl_channel = row
    if tmpl_channel != req.channel:
        return {"status": "error", "reason": "channel_mismatch"}

    # 2. check user preferences
    pref = con.execute("SELECT enabled FROM prefs WHERE user_id=? AND channel=?",
                       (req.user_id, req.channel)).fetchone()
    if pref and pref[0] == 0:
        return {"status": "skipped", "reason": "opted_out"}

    # 3. render
    body = render(body_tmpl, req.vars)

    # 4. idempotency: INSERT OR IGNORE on the UNIQUE dedup_key
    cur = con.execute(
        "INSERT OR IGNORE INTO send_log(user_id,channel,dedup_key,status,ts) "
        "VALUES (?,?,?,?,?)",
        (req.user_id, req.channel, req.dedup_key, "queued", time.time()))
    if cur.rowcount == 0:
        return {"status": "duplicate", "dedup_key": req.dedup_key}
    send_id = cur.lastrowid
    con.commit()

    # 5. deliver (in prod this would be done by a worker reading from a queue)
    result = fake_provider_send(req.channel, req.user_id, body)
    con.execute("UPDATE send_log SET status=? WHERE id=?", (result, send_id))
    con.commit()
    return {"status": result, "send_id": send_id}

# --- try it ---
r1 = NotifyRequest(user_id=42, template="order_shipped", channel="push",
                   dedup_key="order-123-shipped",
                   vars={"name":"Alice","order_id":123})
print("1st call :", notify(r1))
print("2nd call :", notify(r1))   # duplicate — no re-send

r2 = NotifyRequest(user_id=99, template="order_shipped", channel="push",
                   dedup_key="order-999-shipped",
                   vars={"name":"Bob","order_id":999})
print("opted-out:", notify(r2))   # user 99 disabled push

print("\nsend_log contents:")
for row in con.execute("SELECT id,user_id,channel,dedup_key,status FROM send_log"):
    print(" ", row)


## What to notice

- The **`UNIQUE` constraint on `dedup_key`** turns idempotency into a one-liner:
  `INSERT OR IGNORE` + check `rowcount`. No race conditions, no extra Redis call needed
  at this scale.
- Preferences are checked **before** anything expensive (rendering, network).
- Every state transition hits `send_log`, so customer support can always answer
  "what happened to this notification?".

In **Notebook 3** we'll go deeper into the three trickiest pieces:
priority queues, retries with backoff + DLQ, and idempotency — each shown as a
**bad → better → best** progression.